# SENTINELX - Exploratory Data Analysis (EDA)
## Phase 1: Defensive Network Intrusion Detection Machine Learning

### Objective
Understand the distribution, statistical properties, class balance, and feature relationships within network flow telemetry data. In defensive cybersecurity, EDA helps us identify behavioral differences between benign network communications and malicious intrusion patterns (such as DoS Floods, Reconnaissance Port Scans, and Brute Force attacks).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.src.preprocessing.data_loader import DataLoader

print("Libraries loaded successfully.")

### 1. Load and Inspect Raw Network Traffic Data

In [ ]:
data_path = "../data/raw/network_traffic.csv"
loader = DataLoader(data_path)
df = loader.load_csv()

inspection = loader.inspect_dataset(df)
print(f"Total Flow Records : {inspection['total_rows']:,}")
print(f"Total Features     : {inspection['total_columns']}")
print(f"Missing Values     : {inspection['total_missing_cells']}")
print(f"Duplicates         : {inspection['duplicate_rows']}")

df.head()

### 2. Class Distribution & Imbalance Analysis
In cybersecurity, normal traffic heavily dominates real network environments. Let's inspect class frequencies.

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df, x="label", palette="viridis")
plt.title("Network Traffic Class Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Traffic Type", fontsize=10)
plt.ylabel("Flow Count", fontsize=10)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()

### 3. Protocol & Connection Flag Distributions by Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x="protocol_type", hue="label", ax=axes[0], palette="Set2")
axes[0].set_title("Protocol Distribution by Traffic Class", fontweight="bold")

sns.countplot(data=df, x="flag", hue="label", ax=axes[1], palette="Set1")
axes[1].set_title("Connection Status Flag by Traffic Class", fontweight="bold")
plt.tight_layout()
plt.show()

### 4. Numerical Feature Differences (e.g. Byte Counts and Connection Rates)

In [ ]:
df.groupby("label")[["duration", "src_bytes", "dst_bytes", "count", "srv_count"]].mean()

### Summary of Key EDA Findings for Defensive ML Modeling
1. **Flag Signatures**: SYN flood attacks exhibit high `S0` flag concentrations with `0` destination bytes.
2. **Connection Rates**: DoS attacks spike the short-window `count` parameter significantly compared to regular browsing.
3. **Port Scans**: High `diff_srv_rate` and `REJ` flags indicate multi-port probing behavior.
4. **Stratified Splitting**: Because malicious flows are fewer than normal traffic, stratified train-test splitting is mandatory to preserve class distributions.